In [1]:
import numpy as np
import pandas as pd
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn import svm
from sklearn.metrics import classification_report
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (precision_score, recall_score,f1_score, accuracy_score,mean_squared_error,mean_absolute_error)
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import Normalizer
from pandas_ml import ConfusionMatrix

In [2]:
train = pd.read_csv('UNSW_NB15_training-set.csv')
test = pd.read_csv('UNSW_NB15_testing-set.csv')
combined_data = pd.concat([train, test]).drop(['id'],axis=1)

In [3]:
# Contaminsation mean pollution (outliers) in data
tmp = train.where(train['attack_cat'] == "Normal").dropna()
contamination = round(1 - len(tmp)/len(train), 2)
print("train contamination ", contamination)

tmp = test.where(test['attack_cat'] == "Normal").dropna()
print("test  contamination ", round(1 - len(tmp)/len(test),2),'\n')

if contamination > 0.5:
    print(f'contamination is {contamination}, which is greater than 0.5. Fixing...')
    contamination = round(1-contamination,2)
    print(f'contamination is now {contamination}')

train contamination  0.55
test  contamination  0.68 

contamination is 0.55, which is greater than 0.5. Fixing...
contamination is now 0.45


In [4]:
from sklearn.preprocessing import LabelEncoder,normalize
le1 = LabelEncoder()
le = LabelEncoder()

vector = combined_data['attack_cat']

print("attack cat:", set(list(vector))) # use print to make it print on single line 

combined_data['attack_cat'] = le1.fit_transform(vector)
combined_data['proto'] = le.fit_transform(combined_data['proto'])
combined_data['service'] = le.fit_transform(combined_data['service'])
combined_data['state'] = le.fit_transform(combined_data['state'])

vector = combined_data['attack_cat']
print('\nDescribing attack_type: ')
print("min", vector.min())
print("max", vector.max())
print("mode",vector.mode(), "Which is,", le1.inverse_transform(vector.mode()))
print("mode", len(np.where(vector.values==6)[0])/len(vector),"%")

attack cat: {'Backdoor', 'DoS', 'Worms', 'Exploits', 'Normal', 'Reconnaissance', 'Fuzzers', 'Generic', 'Analysis', 'Shellcode'}

Describing attack_type: 
min 0
max 9
mode 0    6
dtype: int32 Which is, ['Normal']
mode 0.3609225646458884 %


In [5]:
le1.inverse_transform([0,1,2,3,4,5,6,7,8,9])
combined_data.head(3)

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,0.000011,119,0,5,2,0,496,0,90909.0902,254,...,1,2,0,0,0,1,2,0,6,0
1,0.000008,119,0,5,2,0,1762,0,125000.0003,254,...,1,2,0,0,0,1,2,0,6,0
2,0.000005,119,0,5,2,0,1068,0,200000.0051,254,...,1,3,0,0,0,1,3,0,6,0


In [6]:
## OMITTED: For statistical feature removal

lowSTD = list(combined_data.std().to_frame().nsmallest(6, columns=0).index)
# this is stupid. suppose a feature has a 1.0 (spearman or pearson) correlation, OR conditional probability, when not 0.... That a very useful feature  

lowCORR = list(combined_data.corr().abs().sort_values('attack_cat')['attack_cat'].nsmallest(3).index) # .where(lambda x: x < 0.005).dropna()
# This might be stupid. A Deep MLP (feed forward neural net) may see patterns

drop = set( lowCORR + lowSTD)
drop = {'ackdat', 'ct_ftp_cmd', 'djit', 'is_ftp_login', 'is_sm_ips_ports', 'response_body_len', 'sjit', 'synack', 'tcprtt'}
# print(f'Before {combined_data.shape}')
combined_data_reduced=combined_data # .drop(drop,axis=1)
# print(f'After {combined_data.shape}')

In [7]:
data_x = combined_data_reduced.drop(['attack_cat','label'], axis=1) # droped label
data_y = combined_data_reduced.loc[:,['label']]
# del combined_data # free mem
X_train, X_test, y_train, y_test = train_test_split(data_x, data_y, test_size=.20, random_state=42) # TODO

In [14]:

y_train.shape
X_test.shape # test is larger... good 
y_test.shape
X_train.shape

(206138, 42)

In [15]:
# traindata = pd.read_csv('UNSW_NB15_training-set.csv', header=None)
# testdata = pd.read_csv('UNSW_NB15_testing-set.csv', header=None)
# traindata = pd.read_csv('kddtrain.csv', header=None)
# testdata = pd.read_csv('kddtest.csv', header=None)

# X = traindata.iloc[:,1:42]
# Y = traindata.iloc[:,0]
# C = testdata.iloc[:,0]
# T = testdata.iloc[:,1:42]
X = X_train
Y = y_train
C = y_test
T = X_test

scaler = Normalizer().fit(X)
trainX = scaler.transform(X)

scaler = Normalizer().fit(T)
testT = scaler.transform(T)


traindata = np.array(trainX)
trainlabel = np.array(Y)

testdata = np.array(testT)
testlabel = np.array(C)


model = LogisticRegression()
model.fit(traindata, trainlabel)


# make predictions
expected = testlabel
predicted = model.predict(testdata)

#predicted = predicted.reshape(len(predicted),1)
print(expected.shape)
print(predicted.shape)

print("***************************************************************")


D:\Anaconda3\envs\TF_36m\lib\site-packages\sklearn\linear_model\logistic.py:432: FutureWarning: Default solver will be changed to 'lbfgs' in 0.22. Specify a solver to silence this warning.
  FutureWarning)
D:\Anaconda3\envs\TF_36m\lib\site-packages\sklearn\utils\validation.py:752: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


(51535, 1)
(51535,)
***************************************************************


In [16]:
from keras.models import Sequential, Model
from keras.layers import Dense, Dropout, Activation, Embedding
from keras.wrappers.scikit_learn import KerasClassifier
import h5py
from keras import callbacks
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
from keras.utils import to_categorical
def build_model():
    # 1. define the network
    model = Sequential()
    #model = Model()
    model.add(Dense(1024,input_dim=42,activation='relu'))  
    model.add(Dropout(0.01))
    model.add(Dense(1))
    model.add(Activation('sigmoid'))
    # try using different optimizers and different optimizer configs
    model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])   
   # model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])   
    return model

In [23]:
#DNN
checkpointer = callbacks.ModelCheckpoint(filepath="./DNNResult/checkpoint-{epoch:02d}.hdf5", verbose=1, save_best_only=True, monitor='loss')
#csv_logger = CSVLogger('./DNNResult/training_set_dnnanalysis.csv',separator=',', append=False)
model = KerasClassifier(build_fn=build_model, epochs=100, batch_size=84)
#model.fit(traindata, trainlabel, callbacks=[checkpointer,csv_logger])
model.fit(traindata, trainlabel, callbacks=[checkpointer])
#model.save("DNNResult/dnn1layer_model.hdf5")

# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model
expected = expected.flatten()
predicted = predicted.flatten()
print(predicted.shape)
print(expected.shape)

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("***************************************************************")


Epoch 1/100
206138/206138 [==============================] - 10s 46us/step - loss: 0.4444 - acc: 0.7493

Epoch 00001: loss improved from inf to 0.44445, saving model to ./DNNResult/checkpoint-01.hdf5
Epoch 2/100
206138/206138 [==============================] - 9s 42us/step - loss: 0.4288 - acc: 0.7673

Epoch 00002: loss improved from 0.44445 to 0.42875, saving model to ./DNNResult/checkpoint-02.hdf5
Epoch 3/100
206138/206138 [==============================] - 9s 44us/step - loss: 0.4178 - acc: 0.7843

Epoch 00003: loss improved from 0.42875 to 0.41775, saving model to ./DNNResult/checkpoint-03.hdf5
Epoch 4/100
206138/206138 [==============================] - 9s 44us/step - loss: 0.4074 - acc: 0.7959

Epoch 00004: loss improved from 0.41775 to 0.40744, saving model to ./DNNResult/checkpoint-04.hdf5
Epoch 5/100
206138/206138 [==============================] - 9s 44us/step - loss: 0.3982 - acc: 0.8038

Epoch 00005: loss improved from 0.40744 to 0.39823, saving model to ./DNNResult/checkpo

206138/206138 [==============================] - 9s 45us/step - loss: 0.3537 - acc: 0.8210

Epoch 00042: loss improved from 0.35505 to 0.35369, saving model to ./DNNResult/checkpoint-42.hdf5
Epoch 43/100
206138/206138 [==============================] - 9s 44us/step - loss: 0.3532 - acc: 0.8215

Epoch 00043: loss improved from 0.35369 to 0.35315, saving model to ./DNNResult/checkpoint-43.hdf5
Epoch 44/100
206138/206138 [==============================] - 9s 44us/step - loss: 0.3529 - acc: 0.8205

Epoch 00044: loss improved from 0.35315 to 0.35289, saving model to ./DNNResult/checkpoint-44.hdf5
Epoch 45/100
206138/206138 [==============================] - 9s 45us/step - loss: 0.3522 - acc: 0.8204

Epoch 00045: loss improved from 0.35289 to 0.35219, saving model to ./DNNResult/checkpoint-45.hdf5
Epoch 46/100
206138/206138 [==============================] - 9s 45us/step - loss: 0.3512 - acc: 0.8207

Epoch 00046: loss improved from 0.35219 to 0.35121, saving model to ./DNNResult/checkpoint-4

206138/206138 [==============================] - 8s 40us/step - loss: 0.3230 - acc: 0.8224

Epoch 00084: loss improved from 0.32345 to 0.32298, saving model to ./DNNResult/checkpoint-84.hdf5
Epoch 85/100
206138/206138 [==============================] - 8s 40us/step - loss: 0.3217 - acc: 0.8226

Epoch 00085: loss improved from 0.32298 to 0.32171, saving model to ./DNNResult/checkpoint-85.hdf5
Epoch 86/100
206138/206138 [==============================] - 8s 40us/step - loss: 0.3224 - acc: 0.8234

Epoch 00086: loss did not improve from 0.32171
Epoch 87/100
206138/206138 [==============================] - 8s 40us/step - loss: 0.3221 - acc: 0.8234

Epoch 00087: loss did not improve from 0.32171
Epoch 88/100
206138/206138 [==============================] - 8s 40us/step - loss: 0.3222 - acc: 0.8227

Epoch 00088: loss did not improve from 0.32171
Epoch 89/100
206138/206138 [==============================] - 8s 40us/step - loss: 0.3214 - acc: 0.8239

Epoch 00089: loss improved from 0.32171 to 0